# In-App Bug Capture Widget (`window.craftBugCapture`)

Toggleable help icon (controlled via app menu / command launcher setting) that opens a bug-report modal, collects user description, URL, viewport, recent console errors, best-effort screenshot, and serialized DOM, and submits them to the `craft-support` ingestion endpoint (which sanitizes HTML with `fast5ever` before uploading to Google Drive via `solvedrive`).

## Templates (`#| export_ft`)

In [ ]:
#| export_ft CRAFT.js
#| eval: false
TEMPLATES = {
    'bug_help_btn': Button(
        svg_icon('M9.09 9a3 3 0 0 1 5.83 1c0 2-3 3-3 3M12 17h.01M22 12a10 10 0 1 1-20 0 10 10 0 0 1 20 0Z', size=16),
        type='button', id='craft-bug-help-btn', cls='craft-bug-help-btn',
        title='Report a bug (solveitcrafts support)', aria_label='Report a bug',
    ),
    'bug_modal': craft_overlay(
        craft_card(
            Div(
                Span('Report a Bug', cls='craft-bug-title'),
                Button('×', type='button', cls='craft-bug-close', aria_label='Close', data_ref='close'),
                cls='craft-bug-hdr',
            ),
            Form(
                Label('Title', craft_input(name='title', placeholder='Short summary of the issue', required=True, data_ref='title')),
                Label('Description', Textarea(name='description', rows='4', cls='uk-textarea craft-bug-ta', placeholder='What happened? What did you expect?', required=True, data_ref='description')),
                Label('Reporter notes (optional)', Textarea(name='reporter_notes', rows='2', cls='uk-textarea craft-bug-ta', placeholder='Steps to reproduce or extra context', data_ref='notes')),
                Label('Ingestion endpoint', craft_input(name='endpoint', placeholder='http://127.0.0.1:8791/api/tickets', data_ref='endpoint')),
                Div(
                    Label(Input(type='checkbox', checked=True, data_ref='inc_dom'), ' Include sanitized DOM snapshot', cls='craft-bug-chk'),
                    Label(Input(type='checkbox', checked=True, data_ref='inc_shot'), ' Include best-effort screenshot', cls='craft-bug-chk'),
                    cls='craft-bug-opts',
                ),
                Div('', cls='craft-bug-meta', data_ref='meta'),
                Div('', cls='craft-bug-status', data_ref='status'),
                Div(
                    craft_btn('Cancel', variant='default', data_ref='cancel'),
                    craft_btn('Send Report', variant='primary', type='submit', data_ref='submit'),
                    cls='craft-bug-actions',
                ),
                data_ref='form',
            ),
            cls='craft-bug-card',
        ),
        id='craft-bug-overlay', role='dialog', aria_modal='true', aria_label='Report a bug',
    ),
}

## Styles (`#| export_css`)

In [ ]:
#| export_css CRAFT.css
#| eval: false
.craft-bug-help-btn {
  position: fixed;
  right: 16px;
  bottom: 18px;
  z-index: 950;
  width: 34px;
  height: 34px;
  border-radius: 999px;
  border: 1px solid var(--craft-border, #2f334d);
  background: var(--craft-surface, #16161e);
  color: var(--craft-fg, #cfc9c2);
  display: inline-flex;
  align-items: center;
  justify-content: center;
  cursor: pointer;
  box-shadow: 0 4px 14px rgba(0, 0, 0, 0.35);
  opacity: 0.85;
  transition: opacity 120ms ease, transform 120ms ease;
}
.craft-bug-help-btn:hover { opacity: 1; transform: translateY(-1px); color: var(--craft-accent, #7aa2f7); }
.craft-bug-help-btn[hidden] { display: none !important; }
#craft-bug-overlay { align-items: center; justify-content: center; z-index: 1100; }
#craft-bug-overlay.open { display: flex; }
.craft-bug-card { width: min(520px, 92vw); max-height: 88vh; overflow-y: auto; padding: 14px 16px; }
.craft-bug-hdr { display: flex; align-items: center; justify-content: space-between; margin-bottom: 10px; }
.craft-bug-title { font-weight: 600; font-size: 13px; }
.craft-bug-close { background: none; border: none; color: inherit; font-size: 18px; cursor: pointer; line-height: 1; }
.craft-bug-card label { display: block; font-size: 11px; margin-bottom: 8px; color: var(--craft-muted, #9aa5ce); }
.craft-bug-card input[type="text"], .craft-bug-ta { width: 100%; margin-top: 3px; font-size: 12px; }
.craft-bug-opts { display: flex; gap: 14px; flex-wrap: wrap; margin: 6px 0 8px; }
.craft-bug-chk { display: inline-flex !important; align-items: center; gap: 5px; margin: 0 !important; cursor: pointer; }
.craft-bug-meta { font-size: 11px; color: var(--craft-muted, #9aa5ce); margin-bottom: 6px; }
.craft-bug-status { font-size: 11px; min-height: 16px; margin-bottom: 8px; word-break: break-word; }
.craft-bug-status.ok { color: var(--craft-green, #9ece6a); }
.craft-bug-status.err { color: var(--craft-red, #f7768e); }
.craft-bug-actions { display: flex; justify-content: flex-end; gap: 8px; }

## Runtime (`#| export_js_iife`)

In [ ]:
#| export_js_iife CRAFT.js craftBugCapture
#| eval: false
window.craftBugCapture?.disable?.();

const CORE = window.__craftCMCore;
if (!CORE) return;
const {on, store, mountTpl, mountModal, notify, kid, onDomChange} = CORE;
const ac = new AbortController(), sig = {signal: ac.signal};

const ENABLED_KEY = 'craft-bug-capture:enabled';
const ENDPOINT_KEY = 'craft-bug-capture:endpoint';
const DEFAULT_ENDPOINT = window.CRAFT_SUPPORT_ENDPOINT || 'http://127.0.0.1:8791/api/tickets';
const MAX_ERRORS = 50;

const recentErrors = [];
const pushErr = (msg) => {
  const text = String(msg || '').slice(0, 600);
  if (!text) return;
  recentErrors.push(`[${new Date().toISOString()}] ${text}`);
  if (recentErrors.length > MAX_ERRORS) recentErrors.shift();
};

const isEnabled = () => store.get(ENABLED_KEY) === '1';
const getEndpoint = () => store.get(ENDPOINT_KEY) || DEFAULT_ENDPOINT;

let origConsoleError = null;
let wrappedConsoleError = null;
let errAc = null;

function uninstallErrorCapture() {
  errAc?.abort();
  errAc = null;
  if (wrappedConsoleError && console.error === wrappedConsoleError && origConsoleError) {
    console.error = origConsoleError;
  }
  wrappedConsoleError = null;
  origConsoleError = null;
}

function syncErrorCapture() {
  if (!isEnabled()) { uninstallErrorCapture(); return; }
  if (wrappedConsoleError) return;
  origConsoleError = console.error;
  const boundOrig = origConsoleError.bind(console);
  wrappedConsoleError = (...args) => {
    try {
      pushErr(args.map(a => (a instanceof Error ? (a.stack || a.message) : typeof a === 'object' ? JSON.stringify(a) : String(a))).join(' '));
    } catch (_) {}
    return boundOrig(...args);
  };
  console.error = wrappedConsoleError;
  errAc = new AbortController();
  const esig = {signal: errAc.signal};
  on(window, 'error', (e) => pushErr(e?.error?.stack || e?.message || 'Window error'), esig);
  on(window, 'unhandledrejection', (e) => pushErr(e?.reason?.stack || e?.reason?.message || String(e?.reason || 'Unhandled rejection')), esig);
}

const helpBtn = mountTpl('bug_help_btn');
document.body.appendChild(helpBtn);
const syncVisibility = () => { helpBtn.hidden = !isEnabled(); };
syncVisibility();
syncErrorCapture();


### Client-Side DOM Sanitization & Screenshot

In [ ]:
#| export_js_iife CRAFT.js craftBugCapture
#| eval: false
/* Client-Side DOM Sanitization & Best-Effort Screenshot */
const SECRET_TEXT_RE = /\b(?:Bearer\s+[A-Za-z0-9._~+/=-]{12,}|(?:sk|pk|rk)_(?:live|test)?_[A-Za-z0-9]{16,}|gh[pousr]_[A-Za-z0-9]{20,}|AIza[0-9A-Za-z_-]{30,}|eyJ[A-Za-z0-9_-]{10,}\.[A-Za-z0-9_-]{10,}\.[A-Za-z0-9_-]{10,}|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,})\b/g;
const SENSITIVE_ATTR_RE = /token|secret|pass|auth|cookie|session|csrf|jwt|api[_-]?key|bearer/i;
const URL_ATTR_RE = /^(?:href|src|action|formaction|xlink:href|srcset|ping|background)$/;

function redactSecretText(text) {
  const s = String(text ?? '');
  SECRET_TEXT_RE.lastIndex = 0;
  return s.replace(SECRET_TEXT_RE, '[REDACTED]');
}

function isDangerousUrl(val) {
  return /^(?:javascript|vbscript):/i.test(String(val || '').replace(/[\u0000-\u0020]+/g, ''));
}

function serializeCleanDom(maxChars = 500000) {
  const clone = document.documentElement.cloneNode(true);
  clone.querySelectorAll('#craft-bug-overlay, #craft-bug-help-btn, script, noscript, iframe[srcdoc], meta[name*="csrf" i], meta[name*="token" i]').forEach(n => n.remove());
  for (const elNode of clone.querySelectorAll('*')) {
    const tag = elNode.tagName;
    if (tag === 'INPUT' || tag === 'TEXTAREA' || tag === 'SELECT') {
      const type = (elNode.getAttribute('type') || '').toLowerCase();
      const ident = `${elNode.getAttribute('name') || ''} ${elNode.getAttribute('id') || ''}`;
      const redact = type === 'password' || type === 'hidden' || type === 'email' || SENSITIVE_ATTR_RE.test(ident);
      if ('value' in elNode) elNode.value = '';
      if (elNode.hasAttribute('value')) elNode.setAttribute('value', redact ? '[REDACTED]' : '');
      if (tag === 'TEXTAREA') elNode.textContent = redact ? '[REDACTED]' : '';
    }
    for (const attr of [...elNode.attributes]) {
      const an = attr.name.toLowerCase();
      const av = attr.value || '';
      if (an.startsWith('on')) { elNode.removeAttribute(attr.name); continue; }
      if (URL_ATTR_RE.test(an) && isDangerousUrl(av)) { elNode.removeAttribute(attr.name); continue; }
      SECRET_TEXT_RE.lastIndex = 0;
      if (SENSITIVE_ATTR_RE.test(an) || SECRET_TEXT_RE.test(av)) {
        elNode.setAttribute(attr.name, '[REDACTED]');
      }
    }
  }
  const walker = document.createTreeWalker(clone, NodeFilter.SHOW_TEXT);
  let node;
  while ((node = walker.nextNode())) {
    if (node.nodeValue) {
      const cleaned = redactSecretText(node.nodeValue);
      if (cleaned !== node.nodeValue) node.nodeValue = cleaned;
    }
  }
  const html = '<!doctype html>\n' + clone.outerHTML;
  return html.length > maxChars ? html.slice(0, maxChars) + '\n<!-- truncated -->' : html;
}

async function captureBestEffortScreenshot() {
  try {
    if (typeof window.html2canvas === 'function') {
      const canvas = await window.html2canvas(document.body, {logging: false, useCORS: true, scale: 1});
      return canvas.toDataURL('image/png');
    }
    const w = Math.min(window.innerWidth || 1280, 1440);
    const h = Math.min(window.innerHeight || 800, 900);
    const xhtml = new XMLSerializer().serializeToString(document.body);
    const svg = `<svg xmlns="http://www.w3.org/2000/svg" width="${w}" height="${h}"><foreignObject width="100%" height="100%">${xhtml}</foreignObject></svg>`;
    const img = new Image();
    const url = URL.createObjectURL(new Blob([svg], {type: 'image/svg+xml;charset=utf-8'}));
    try {
      await new Promise((res, rej) => { img.onload = res; img.onerror = rej; img.src = url; });
      const canvas = document.createElement('canvas');
      canvas.width = w; canvas.height = h;
      canvas.getContext('2d').drawImage(img, 0, 0);
      return canvas.toDataURL('image/png');
    } finally {
      URL.revokeObjectURL(url);
    }
  } catch (_) {
    return null;
  }
}

### Bug Report Modal & Submission

In [ ]:
#| export_js_iife CRAFT.js craftBugCapture
#| eval: false
/* Bug Report Modal & Submission */
let modal = null;
function closeModal() { modal?.close(); }

function ensureModal() {
  if (modal) return modal;
  modal = mountModal('bug_modal', {signal: ac.signal});
  on(modal.cancel, 'click', closeModal, sig);
  on(modal.form, 'submit', async (e) => {
    e.preventDefault();
    const statusEl = modal.status;
    const submitBtn = modal.submit;
    const endpoint = (modal.endpoint.value || '').trim() || DEFAULT_ENDPOINT;
    store.set(ENDPOINT_KEY, endpoint);
    submitBtn.disabled = true;
    statusEl.className = 'craft-bug-status';
    statusEl.textContent = 'Capturing context and uploading ticket...';
    try {
      const payload = {
        title: modal.title.value.trim(),
        description: modal.description.value.trim(),
        reporter_notes: modal.notes.value.trim(),
        url: location.href,
        viewport: {width: window.innerWidth, height: window.innerHeight},
        user_agent: navigator.userAgent,
        dialog_id: typeof kid === 'function' ? kid() : null,
        console_errors: recentErrors.slice(),
        dom_html: modal.inc_dom.checked ? serializeCleanDom() : '',
        screenshot_data_url: modal.inc_shot.checked ? await captureBestEffortScreenshot() : null,
      };
      const resp = await fetch(endpoint, {
        method: 'POST',
        headers: {'Content-Type': 'application/json'},
        body: JSON.stringify(payload),
      });
      const data = await resp.json().catch(() => ({}));
      if (!resp.ok) throw new Error(data.error || `HTTP ${resp.status}`);
      statusEl.className = 'craft-bug-status ok';
      statusEl.textContent = `Saved ticket ${data.id}` + (data.folder_url ? ` (${data.folder_url})` : '');
      modal.title.value = '';
      modal.description.value = '';
      modal.notes.value = '';
    } catch (err) {
      statusEl.className = 'craft-bug-status err';
      statusEl.textContent = `Failed to submit: ${err?.message || err}`;
    } finally {
      submitBtn.disabled = false;
    }
  }, sig);
  return modal;
}

function openModal() {
  const m = ensureModal();
  m.endpoint.value = getEndpoint();
  m.meta.textContent = `${location.pathname} · ${window.innerWidth}×${window.innerHeight} · ${recentErrors.length} recent console error(s)`;
  m.status.className = 'craft-bug-status';
  m.status.textContent = '';
  m.open();
  m.title.focus();
}

on(helpBtn, 'click', openModal, sig);

### Menu Toggle, Hotkeys & Lifecycle

In [ ]:
#| export_js_iife CRAFT.js craftBugCapture
#| eval: false
/* Menu Toggle, Hotkeys & Lifecycle */
function setEnabled(val) {
  store.set(ENABLED_KEY, val ? '1' : '0');
  syncVisibility();
  syncErrorCapture();
  syncMenuToggle();
  notify(isEnabled() ? 'Bug-report help icon enabled.' : 'Bug-report help icon hidden.', 'primary');
  return isEnabled();
}

function syncMenuToggle() {
  document.querySelectorAll('.craft-bug-menu-toggle input[type="checkbox"]').forEach(chk => {
    chk.checked = isEnabled();
  });
}

function injectMenuToggle() {
  const lists = document.querySelectorAll('.uk-container > nav.monster-navbar .uk-dropdown ul, .uk-container > nav.monster-navbar .uk-drop ul, .uk-navbar-dropdown ul');
  for (const ul of lists) {
    if (ul.querySelector('.craft-bug-menu-toggle')) continue;
    const li = document.createElement('li');
    li.className = 'craft-bug-menu-toggle';
    const lbl = document.createElement('label');
    lbl.style.cssText = 'display:flex;align-items:center;gap:8px;padding:6px 10px;cursor:pointer;font-size:12px;';
    const chk = document.createElement('input');
    chk.type = 'checkbox';
    chk.checked = isEnabled();
    on(chk, 'change', () => setEnabled(chk.checked), sig);
    lbl.append(chk, document.createTextNode('Show bug-report help icon'));
    li.appendChild(lbl);
    ul.appendChild(li);
  }
}
injectMenuToggle();
const stopMenuObs = onDomChange?.(injectMenuToggle);

window.__hotkeyRegister?.({
  id: 'craft-bug-toggle-icon',
  label: 'Toggle bug-report help icon',
  group: 'CRAFT · Support',
  combo: '',
  handler: () => { setEnabled(!isEnabled()); return true; },
});
window.__hotkeyRegister?.({
  id: 'craft-bug-open-form',
  label: 'Report a bug (open support form)',
  group: 'CRAFT · Support',
  combo: '',
  handler: () => { openModal(); return true; },
});

window.craftBugCapture = {
  isEnabled,
  setEnabled,
  open: openModal,
  close: closeModal,
  getRecentErrors: () => recentErrors.slice(),
  disable: () => {
    ac.abort();
    stopMenuObs?.();
    uninstallErrorCapture();
    modal?.destroy();
    modal = null;
    helpBtn.remove();
    document.querySelectorAll('.craft-bug-menu-toggle').forEach(el => el.remove());
    delete window.craftBugCapture;
  },
};

Verify `19_bug_capture.ipynb` exports `window.craftBugCapture`, its templates, CSS, and client-side sanitization helpers.

In [ ]:
import json, os, subprocess
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets

assets = extract_nb_assets(REPO_ROOT/'nbs/craft/19_bug_capture.ipynb')
js = '\n'.join(code for _, code in assets['CRAFT.js'])
css = '\n'.join(code for _, code in assets['CRAFT.css'])
for sym in ('window.craftBugCapture', 'bug_modal', 'bug_help_btn', 'serializeCleanDom', 'mountModal'):
    assert sym in js, f"Missing {sym}"
assert '#craft-bug-overlay.open' in css and '.craft-bug-help-btn' in css

node_script = r"""
const src = process.env.BUG_JS;
const i0 = src.indexOf('const SECRET_TEXT_RE');
const i1 = src.indexOf('async function captureBestEffortScreenshot');
const {redactSecretText, isDangerousUrl} = new Function(src.slice(i0, i1) + '; return {redactSecretText, isDangerousUrl};')();

// 1. Secret redaction catches Bearer, API keys, JWTs, and emails while preserving normal text
const sample = 'User alice@example.com sent Bearer abcdef1234567890 and sk_live_1234567890abcdef1234 with ghp_1234567890123456789012';
const cleaned = redactSecretText(sample);
for (const leak of ['alice@example.com', 'abcdef1234567890', 'sk_live_', 'ghp_']) {
  if (cleaned.includes(leak)) throw new Error('Leaked secret: ' + leak + ' in ' + cleaned);
}
if (!cleaned.startsWith('User [REDACTED] sent [REDACTED]')) throw new Error('Unexpected redaction: ' + cleaned);
if (redactSecretText('Normal bug report #42') !== 'Normal bug report #42') throw new Error('False positive redaction');

// 2. Dangerous URL detection catches control-char/whitespace obfuscation
for (const bad of ['javascript:alert(1)', '  JAVASCRIPT:void(0)', 'java\tscript:alert(1)', 'vb\nscript:msgbox(1)']) {
  if (!isDangerousUrl(bad)) throw new Error('Missed dangerous URL: ' + JSON.stringify(bad));
}
for (const ok of ['https://example.com/path', '/dialog_?name=CRAFT', 'mailto:test@example.com']) {
  if (isDangerousUrl(ok)) throw new Error('Flagged safe URL: ' + ok);
}

process.stdout.write(JSON.stringify({ok: true}));
"""
res = subprocess.run(['node', '-e', node_script], env={**os.environ, 'BUG_JS': js}, capture_output=True, text=True, check=True)
assert json.loads(res.stdout)['ok']